<a href="https://colab.research.google.com/github/Renadth/LLM-application-engineering-/blob/main/homework/day1_real_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1 Homework — Generate One Answer with a Real Model

**Course:** LLM Application Engineering · SDA-AIE-213  
**Instructor:** Meaad Al-Marri  
**Assignment:** Extra homework, Day 1 (real-model demonstration)  
**Mode:** `real_local_model`. FLAN-T5 small runs **locally on CPU** inside Colab. No API key, no paid service, `external_inference_api_calls = 0`.

**Goal:** run a small real model on one synthetic FAQ, follow the flow **input → tokens → model → answer**, and compare it with the core project's `FakeClient`, which returns fixed answers and generates nothing.

> This notebook uses the course's supplied demo code. My work here is running it, reviewing the answer, and explaining the flow. Keep this notebook **separate** from the core `murshid.ipynb` runtime.

In [3]:
# OPTIONAL: use a separate notebook and runtime, never the core project runtime.
# Internet and model/library downloads required; no API key or paid endpoint.
import importlib.util
import subprocess
import sys
# Preserve Colab's preinstalled matched torch/vision/audio stack.
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cpu'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'sentencepiece==0.2.1'])
import torch
print('Torch runtime:', torch.__version__)
print('If Colab requests a restart, restart this optional runtime and run all cells again.')


Torch runtime: 2.14.1+cpu
If Colab requests a restart, restart this optional runtime and run all cells again.


In [4]:
# OPTIONAL REAL-MODEL EXTENSION: run in a separate notebook/runtime.
# Never paste into or replace cells in the required mock notebook.
import json
import time
from pathlib import Path

LIVE_MODEL_ID = 'google/flan-t5-small'
LIVE_MODEL_REVISION = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
LIVE_OUTPUT_DIR = Path('murshid_live_outputs')




In [5]:
def run_live_demo():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LIVE_MODEL_ID, revision=LIVE_MODEL_REVISION,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    # Public synthetic context only; no private application records or identities.
    context = 'You can check your application status using its reference number.'
    question = 'How can I check my application status?'
    prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=48, do_sample=False, num_beams=1)
    answer = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    # Smoke check only: this does NOT establish factual correctness or safety.
    assert isinstance(answer, str) and answer, 'No answer generated; inspect the runtime.'
    report = {
        'mode': 'real_local_model', 'model': LIVE_MODEL_ID,
        'model_revision': LIVE_MODEL_REVISION, 'device': 'cpu',
        'context': context, 'question': question, 'answer': answer,
        'inference_seconds': round(time.perf_counter() - started, 3),
        'input_tokens': int(encoded['input_ids'].shape[-1]),
        'output_tokens_including_special': int(generated.shape[-1]),
        'external_inference_api_calls': 0,
        'review_required': 'Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims.'
    }
    LIVE_OUTPUT_DIR.mkdir(exist_ok=True)
    (LIVE_OUTPUT_DIR / 'live_demo_result.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))
    return report




In [6]:
live_result = run_live_demo()
print('REVIEW: Does the answer say to use the reference number?')
print('Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "device": "cpu",
  "context": "You can check your application status using its reference number.",
  "question": "How can I check my application status?",
  "answer": "Use its reference number",
  "inference_seconds": 1.632,
  "input_tokens": 34,
  "output_tokens_including_special": 6,
  "external_inference_api_calls": 0,
  "review_required": "Check whether the answer tells the user to use the reference number. Reject unsupported fees, dates or status claims."
}
REVIEW: Does the answer say to use the reference number?
Reject invented fees, dates or application statuses. Nonempty text is not a quality pass.


## Reflection

### The flow: input → tokens → model → answer
The prompt combines the context and the question. The tokenizer splits it into **34 tokens**. FLAN-T5 small runs on the CPU and generates new tokens one at a time (**6 including special tokens**, in 0.286 s). The tokenizer then decodes them back into text: `Use its reference number`.

### How this differs from `FakeClient`
In the core project, `FakeClient` doesn't generate anything. It looks the question up in a fixed dictionary and returns a pre-written sentence, so the answer is always identical and needs no model weights. Here, downloaded model weights actually **generate** the answer, so the wording could change with a different prompt or settings, and it could be incomplete or made up. It has to be checked, not trusted just because text was produced.

### My review of the answer
| Question | Answer |
|---|---|
| Observed answer | `Use its reference number` |
| Correct meaning? | Yes. It tells the user to use the reference number |
| Unsupported facts? | None. No invented fees, dates or status |
| Verdict | ✅ Correct and supported, but shorter than the context sentence |

### Limitations
- **One English example** doesn't prove the model is accurate, safe or good at Arabic.
- A real application would still need input validation, output checks and evaluation around the model.
- Inference time is local Colab CPU time. It changes between runs and doesn't predict API latency.